# Notebook 05: Explainable AI (XAI) Analysis (SHAP & LIME)
**GuidedGuard – Explainable AI for Scam-Guided Digital Payment Detection**

--- 
### XAI Phase Objectives:
- Load trained model (`models/saved_model.pkl`), fitted scaler (`models/scaler.pkl`), and metadata (`models/model_metadata.json`).
- Initialize **SHAP (SHapley Additive exPlanations)** engine for global feature attribution and local waterfall/force plots.
- Initialize **LIME (Local Interpretable Model-agnostic Explanations)** tabular surrogate model for local decision rule weights.
- Translate mathematical feature attributions into **Human-Readable English Narratives** (e.g., *'High transaction amount increased fraud risk score by 0.35'*).
- Export all XAI visualization reports, CSV attribution tables, and text summaries to `outputs/reports/xai/`.

> **Strict Boundary Guardrails:**
> - ❌ No model retraining.
> - ❌ No feature matrix modification.
> - ❌ No Streamlit UI code.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

# Add project root to sys.path
sys.path.append(str(Path.cwd().parent))
import config
from models.model_loader import load_artifacts
from utils.data_loader import load_dataset
from explainability.shap_explainer import (
    initialize_shap,
    generate_shap_values,
    generate_summary_plot,
    generate_waterfall_plot,
    generate_human_readable_summary,
    save_shap_visualizations
)
from explainability.lime_explainer import (
    initialize_lime,
    generate_lime_explanation,
    plot_lime_features,
    save_lime_visualization
)

print("XAI Environment initialized successfully.")

# 1. Model & Artifact Verification
Loading `saved_model.pkl`, `scaler.pkl`, `model_metadata.json`, and sample featured dataset.

In [ ]:
# 1. Load Artifacts
model, scaler, metadata = load_artifacts()
print(f"Loaded Trained Model: {type(model).__name__ if model else 'Placeholder Model'}")
print(f"Loaded Scaler: {type(scaler).__name__ if scaler else 'Placeholder Scaler'}")
if metadata:
    print("Model Training Metadata Summary:")
    display(pd.Series(metadata["metrics"]))

# 2. Load Sample Feature Dataset
feat_df = load_dataset(config.PROCESSED_DATA_DIR / "paysim_featured.csv")
target_col = "isFraud" if "isFraud" in feat_df.columns else "fraud_bool"

X_sample = feat_df.drop(columns=[target_col]).head(200)
y_sample = feat_df[target_col].head(200)
print(f"Loaded XAI Feature Sample Matrix Shape: {X_sample.shape}")

# 2. SHAP Global Feature Importance Analysis
Calculating global SHAP attributions and generating global summary plot.

In [ ]:
# Initialize SHAP Explainer
explainer = initialize_shap(model, background_data=X_sample)
shap_values, base_value = generate_shap_values(explainer, X_sample)

print(f"Base Expected Output Value: {base_value:.4f}")
print(f"SHAP Values Matrix Shape: {shap_values.shape}")

# Plot Global Summary Plot
fig_summary = generate_summary_plot(shap_values, X_sample)
plt.show()

# 3. SHAP Local Transaction Waterfall Explanation
Analyzing individual transaction prediction instance (#0).

In [ ]:
# Generate Local Waterfall Plot for Row 0
fig_waterfall = generate_waterfall_plot(explainer, shap_values, X_sample, row_idx=0)
plt.show()

# 4. LIME Local Linear Surrogate Rules Analysis
Generating interpretable rule bounds and local feature weights using LIME.

In [ ]:
# Initialize LIME Explainer
lime_exp = initialize_lime(np.array(X_sample), feature_names=list(X_sample.columns))
lime_rules = generate_lime_explanation(lime_exp, model, X_sample.iloc[[0]])

print("Top Local LIME Decision Rules:")
display(pd.DataFrame(lime_rules, columns=["Local Rule Boundary", "Feature Weight"]))

# Plot LIME Feature Weights
fig_lime = plot_lime_features(lime_rules, title="Local LIME Decision Rules (Transaction Instance #0)")
plt.show()

# 5. Human-Readable Natural Language Narrative Summary
Converting SHAP & LIME values into clear English sentences.

In [ ]:
# Convert Row 0 SHAP values to dictionary
row_0_attributions = dict(zip(X_sample.columns, shap_values[0]))
narratives = generate_human_readable_summary(row_0_attributions)

print("=== HUMAN-READABLE EXPLANATION NARRATIVE FOR TRANSACTION #0 ===")
for idx, sentence in enumerate(narratives, 1):
    print(f"{idx}. {sentence}")

# 6. Export XAI Artifacts & Visualizations (`outputs/reports/xai/`)
Saving PNG figures, CSV contribution tables, and narrative text files.

In [ ]:
# Export SHAP & Narrative artifacts
output_dir = config.OUTPUTS_DIR / "reports" / "xai"
exported_paths = save_shap_visualizations(shap_values, X_sample, output_dir=output_dir)

# Export LIME plot artifact
lime_png_path = save_lime_visualization(lime_rules, output_path=output_dir / "lime_local_explanation.png")
exported_paths["lime_plot_png"] = lime_png_path

print("=== XAI EXPORT COMPLETE ===")
for k, v in exported_paths.items():
    print(f"• {k}: {v}")